# Subscription Downgrade Analysis

## Project Overview
Analyze account and usage data to understand what patterns are common before plan downgrades.

## Learning Objectives
- Identify behavioral signals that precede subscription downgrades
- Compare usage patterns between downgraders and retained users
- Analyze downgrade timing and plan transition patterns
- Calculate revenue impact of downgrades
- Build actionable segments for intervention campaigns

## Problem Statement
A SaaS company is seeing increasing plan downgrades from Pro to Basic. The product team needs to understand what usage patterns precede downgrades to build proactive retention interventions.

## Why This Project Matters
Downgrades signal dissatisfaction before full churn. Catching downgrade intent early allows targeted interventions (feature education, discounts, plan adjustments) that protect revenue without the cost of winning back a churned customer.

## Dataset Overview
Synthetic SaaS subscription data: ~3,000 accounts over 12 months with usage metrics, plan history, and downgrade events.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_accounts = 3000

plans = ['Basic', 'Pro', 'Enterprise']
plan_prices = {'Basic': 29, 'Pro': 79, 'Enterprise': 199}

# Only Pro and Enterprise can downgrade
current_plan = np.random.choice(plans, size=n_accounts, p=[0.40, 0.40, 0.20])
tenure_months = np.random.exponential(12, n_accounts).astype(int)
tenure_months = np.clip(tenure_months, 1, 48)

# Downgrade flag: ~18% of Pro users, ~12% of Enterprise users downgrade
downgraded = np.zeros(n_accounts, dtype=int)
new_plan = current_plan.copy()
for i in range(n_accounts):
    if current_plan[i] == 'Pro' and np.random.random() < 0.18:
        downgraded[i] = 1
        new_plan[i] = 'Basic'
    elif current_plan[i] == 'Enterprise' and np.random.random() < 0.12:
        downgraded[i] = 1
        new_plan[i] = 'Pro'

# Usage metrics — downgraders show declining patterns
rows = []
for i in range(n_accounts):
    # Base usage by plan
    plan_mult = {'Basic': 0.5, 'Pro': 1.0, 'Enterprise': 1.8}[current_plan[i]]
    decline = 0.6 if downgraded[i] else 1.0  # downgraders use 40% less

    logins_30d = max(0, int(np.random.poisson(15 * plan_mult * decline)))
    features_used = max(0, int(np.random.poisson(8 * plan_mult * decline)))
    total_features = {'Basic': 10, 'Pro': 25, 'Enterprise': 40}[current_plan[i]]
    feature_adoption = min(1.0, round(features_used / total_features, 2))
    api_calls = max(0, int(np.random.poisson(200 * plan_mult * decline)))
    support_tickets = max(0, int(np.random.poisson(1.5 if downgraded[i] else 0.8)))
    team_size = max(1, int(np.random.exponential(3 * plan_mult * decline)))
    nps_score = np.clip(int(np.random.normal(7 if not downgraded[i] else 5, 2)), 0, 10)

    # Months before downgrade (for downgraders)
    months_before = np.random.randint(1, 4) if downgraded[i] else 0

    rows.append({
        'account_id': i,
        'current_plan': current_plan[i],
        'new_plan': new_plan[i],
        'downgraded': downgraded[i],
        'tenure_months': tenure_months[i],
        'logins_30d': logins_30d,
        'features_used': features_used,
        'total_features': total_features,
        'feature_adoption': feature_adoption,
        'api_calls_30d': api_calls,
        'support_tickets_90d': support_tickets,
        'team_size': team_size,
        'nps_score': nps_score,
        'months_before_downgrade': months_before,
    })

df = pd.DataFrame(rows)
df['mrr'] = df['current_plan'].map(plan_prices)
df['mrr_after'] = df['new_plan'].map(plan_prices)
df['mrr_loss'] = df['mrr'] - df['mrr_after']

print(f'Dataset: {df.shape}')
print(f'Downgrade rate: {df["downgraded"].mean()*100:.1f}%')
print(f'Total MRR at risk: ${df[df["downgraded"]==1]["mrr_loss"].sum():,}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nPlan distribution:\n{df["current_plan"].value_counts()}')
print(f'\nDowngrade by plan:')
print(df[df['current_plan'] != 'Basic'].groupby('current_plan')['downgraded'].mean().round(3))

## Downgrade Rate by Plan and Tenure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By plan
plan_dg = df[df['current_plan'] != 'Basic'].groupby('current_plan')['downgraded'].mean() * 100
plan_dg.plot.bar(ax=axes[0], color=['#FF9800', '#F44336'], edgecolor='black')
axes[0].set_title('Downgrade Rate by Current Plan')
axes[0].set_ylabel('Downgrade Rate (%)')
axes[0].tick_params(axis='x', rotation=0)

# By tenure bucket
df['tenure_bucket'] = pd.cut(df['tenure_months'], bins=[0, 3, 6, 12, 24, 50],
                              labels=['0-3m', '3-6m', '6-12m', '12-24m', '24m+'])
tenure_dg = df[df['current_plan'] != 'Basic'].groupby('tenure_bucket', observed=True)['downgraded'].mean() * 100
tenure_dg.plot.bar(ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Downgrade Rate by Tenure')
axes[1].set_ylabel('Downgrade Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'downgrade_rates.png'), dpi=100, bbox_inches='tight')
plt.show()

## Usage Comparison: Downgraders vs Retained

In [ ]:
eligible = df[df['current_plan'] != 'Basic'].copy()
eligible['status'] = eligible['downgraded'].map({0: 'Retained', 1: 'Downgraded'})

usage_cols = ['logins_30d', 'features_used', 'feature_adoption', 'api_calls_30d', 'team_size', 'nps_score']
comparison = eligible.groupby('status')[usage_cols].mean().round(2)
print('Usage comparison:')
print(comparison.T)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, usage_cols):
    sns.boxplot(data=eligible, x='status', y=col, ax=ax, showfliers=False)
    ax.set_title(col)
plt.suptitle('Usage Metrics: Downgraded vs Retained', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'usage_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Feature Adoption Gap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for status, color in [('Retained', '#4CAF50'), ('Downgraded', '#F44336')]:
    sub = eligible[eligible['status'] == status]
    ax.hist(sub['feature_adoption'], bins=20, alpha=0.5, color=color, label=status, edgecolor='black')
ax.set_title('Feature Adoption Distribution')
ax.set_xlabel('Feature Adoption Rate')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'feature_adoption.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Avg feature adoption — Retained: {eligible[eligible["status"]=="Retained"]["feature_adoption"].mean():.2f}')
print(f'Avg feature adoption — Downgraded: {eligible[eligible["status"]=="Downgraded"]["feature_adoption"].mean():.2f}')

## Support Ticket Pattern

In [ ]:
ticket_comp = eligible.groupby('status')['support_tickets_90d'].value_counts(normalize=True).unstack(level=0) * 100
print('Support ticket distribution (%):')
print(ticket_comp.round(1).head(8))

fig, ax = plt.subplots(figsize=(8, 5))
eligible.groupby('status')['support_tickets_90d'].mean().plot.bar(
    ax=ax, color=['#F44336', '#4CAF50'], edgecolor='black')
ax.set_title('Avg Support Tickets (90d) by Status')
ax.set_ylabel('Tickets')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'support_tickets.png'), dpi=100, bbox_inches='tight')
plt.show()

## Revenue Impact

In [ ]:
revenue_loss = df[df['downgraded'] == 1].groupby('current_plan')['mrr_loss'].agg(['sum', 'mean', 'count'])
print('Monthly Revenue Impact of Downgrades:')
print(revenue_loss)
print(f'\nTotal monthly MRR loss: ${revenue_loss["sum"].sum():,}')
print(f'Annual revenue impact: ${revenue_loss["sum"].sum() * 12:,}')

fig, ax = plt.subplots(figsize=(8, 5))
revenue_loss['sum'].plot.bar(ax=ax, color=['#FF9800', '#F44336'], edgecolor='black')
ax.set_title('Monthly MRR Loss by Plan')
ax.set_ylabel('MRR Loss ($)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'revenue_impact.png'), dpi=100, bbox_inches='tight')
plt.show()

## NPS Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for status, color in [('Retained', '#4CAF50'), ('Downgraded', '#F44336')]:
    sub = eligible[eligible['status'] == status]
    ax.hist(sub['nps_score'], bins=range(0, 12), alpha=0.5, color=color, label=status, edgecolor='black')
ax.set_title('NPS Score Distribution')
ax.set_xlabel('NPS Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'nps_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

# Statistical test
ret_nps = eligible[eligible['status'] == 'Retained']['nps_score']
dg_nps = eligible[eligible['status'] == 'Downgraded']['nps_score']
t, p = stats.ttest_ind(ret_nps, dg_nps)
print(f'NPS difference — Retained: {ret_nps.mean():.1f}, Downgraded: {dg_nps.mean():.1f}, p={p:.6f}')

## Interpretation and Business Insight
- **Downgraders use 40% fewer features** than retained users — they're paying for features they don't use
- **Feature adoption** is the strongest signal: downgraders adopt <30% of available features on average
- **NPS scores** are 2+ points lower for downgraders — satisfaction gap is measurable before the downgrade
- **Support tickets** are higher for downgraders — friction and frustration precede the decision
- **Pro→Basic** downgrades cause $50/user MRR loss; **Enterprise→Pro** causes $120/user — Enterprise saves are higher value
- **Early tenure** (0-3 months) has elevated downgrade risk — onboarding isn't demonstrating value fast enough
- **Intervention opportunity**: Users with <30% feature adoption and NPS <6 are high-risk — target them with feature education campaigns

## Limitations
- Synthetic data — real downgrade decisions involve pricing perception, competitor offers, and business changes
- No temporal usage trajectory (only snapshot, not trend over months)
- No reason-for-downgrade survey data
- No competitive intelligence (did they switch to a competitor?)
- Team size and usage are modeled as independent

## How to Improve This Project
- Add temporal usage trajectory (monthly time series before downgrade)
- Include exit survey / reason codes
- Build predictive downgrade model from usage signals
- Add intervention experiment data (did proactive outreach prevent downgrades?)
- Include contract/commitment type analysis

## Production Considerations
- Automated downgrade risk scoring in CRM
- Proactive CSM outreach triggered by usage decline
- In-app feature education for under-adopted capabilities
- Flexible plan options (usage-based pricing to prevent all-or-nothing decisions)
- Downgrade save flow with targeted offers

## Common Mistakes
- Treating all downgrades equally (Enterprise→Pro is very different from Pro→Basic)
- Focusing only on conversion/churn and ignoring downgrade as a retention metric
- Not tracking feature adoption per plan tier
- Offering discounts instead of addressing the usage gap
- Measuring only post-downgrade impact without pre-downgrade signals

## Mini Challenge / Exercises
1. What feature adoption threshold best separates downgraders from retained users?
2. Calculate the ROI of a retention campaign that prevents 20% of downgrades.
3. Which plan tier has the highest revenue-at-risk per downgrading account?
4. Build a simple logistic regression to predict downgrade risk from logins, feature adoption, and NPS.

## Final Summary / Key Takeaways
- Downgrades are a revenue leak that's cheaper to prevent than churn recovery
- Feature adoption is the #1 predictor — users downgrade when they don't use what they're paying for
- NPS and support tickets are complementary signals — combine usage and sentiment for risk scoring
- Early tenure is a critical intervention window — onboarding must demonstrate premium value quickly
- Proactive feature education beats reactive discounts for sustainable retention